# Model Evaluation & Performance Improvement
**Dataset:** Breast Cancer Wisconsin Diagnostic Dataset (Preprocessed & Engineered)

### Notebook Objectives:
1. **Comprehensive Evaluation Metrics:** Compute **Accuracy**, **Precision**, **Recall**, **F1-Score**, **ROC-AUC**, and **Confusion Matrices**.
2. **Multi-Model Comparison:** Compare 8 classifiers across linear, kernel, instance-based, single-tree, and ensemble families.
3. **Overfitting vs. Underfitting Diagnosis:** Compare Training vs. 5-Fold Cross-Validation vs. Held-Out Test accuracy and plot Learning Curves.
4. **Performance Improvement:** Demonstrate how regularization (`L2`), depth constraints, and ensemble averaging resolve underfitting and overfitting.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, learning_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Load Model-Ready Dataset & Create Stratified 80/20 Train-Test Split

In [ ]:
data_path = 'data/breast_cancer_model_ready.csv'
if not os.path.exists(data_path) and os.path.exists('../data/breast_cancer_model_ready.csv'):
    data_path = '../data/breast_cancer_model_ready.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['area_perimeter_ratio'] = df['mean area'] / (df['mean perimeter'] + 1e-6)
    df['radius_worst_to_mean_ratio'] = df['worst radius'] / (df['mean radius'] + 1e-6)
    df['concavity_smoothness_index'] = df['mean concavity'] * df['mean smoothness']
    df['circularity_deviation'] = np.abs((df['mean perimeter'] / (2 * np.pi * df['mean radius'] + 1e-6)) - 1.0)
    df['nuclear_malformation_score'] = df['worst concavity'] + df['worst concave points'] + df['worst compactness']
    df['target'] = data.target

X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print(f'Training Set: {X_train.shape} | Held-Out Test Set: {X_test.shape}')

## 2. Evaluate Candidate Models & Diagnose Overfitting vs. Underfitting
We intentionally include:
- **Underfitting baselines:** `DummyClassifier` and `DecisionTreeClassifier(max_depth=1)` (high bias, low accuracy on both train and test).
- **Overfitting model:** `DecisionTreeClassifier(max_depth=None)` (`100%` train accuracy vs. `91.23%` test accuracy — an `8.77%` generalization gap).
- **Well-generalized models:** Regularized `SVM (RBF)`, `Logistic Regression (Tuned L2)`, `KNN`, `Random Forest`, and `Gradient Boosting`.

In [ ]:
models = {
    'Dummy Baseline (Underfit)': DummyClassifier(strategy='most_frequent'),
    'Decision Tree (Depth=1, Underfit)': DecisionTreeClassifier(max_depth=1, random_state=42),
    'Decision Tree (Unconstrained, Overfit)': DecisionTreeClassifier(max_depth=None, random_state=42),
    'K-Nearest Neighbors (k=7)': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=7))]),
    'Random Forest (n=150)': RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    'Support Vector Machine (RBF)': Pipeline([('scaler', StandardScaler()), ('clf', SVC(C=2.0, kernel='rbf', probability=True, random_state=42))]),
    'Logistic Regression (Tuned L2)': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(C=0.5, penalty='l2', solver='liblinear', random_state=42))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
fitted_models = {}

for name, model in models.items():
    cv_res = cross_validate(model, X_train, y_train, cv=cv, scoring=['accuracy', 'f1'])
    model.fit(X_train, y_train)
    fitted_models[name] = model

    train_acc = accuracy_score(y_train, model.predict(X_train))
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    test_acc = accuracy_score(y_test, y_pred)
    gap = train_acc - test_acc

    if 'Dummy' in name or 'Depth=1' in name:
        status = 'Underfitting (High Bias)'
    elif gap >= 0.04:
        status = 'Overfitting (High Variance)'
    else:
        status = 'Well-Generalized (Balanced)'

    rows.append({
        'Model': name,
        'Train Accuracy': round(train_acc, 4),
        '5-Fold CV Accuracy': round(cv_res['test_accuracy'].mean(), 4),
        'Test Accuracy': round(test_acc, 4),
        'Train-Test Gap': round(gap, 4),
        'Test Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Test Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'Test F1-Score': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'Test ROC-AUC': round(roc_auc_score(y_test, y_prob), 4),
        'Fit Diagnosis': status
    })

comp_df = pd.DataFrame(rows).sort_values(by='Test F1-Score', ascending=False)
display(comp_df)

## 3. Visual Diagnostics: Overfitting/Underfitting Chart, ROC Curves & Learning Curve

In [ ]:
# 1. Train vs CV vs Test Accuracy Bar Chart
plt.figure(figsize=(11, 5.5))
plot_melt = comp_df.melt(id_vars='Model', value_vars=['Train Accuracy', '5-Fold CV Accuracy', 'Test Accuracy'], var_name='Split', value_name='Accuracy')
sns.barplot(data=plot_melt, x='Model', y='Accuracy', hue='Split', palette=['#95a5a6', '#3498db', '#2ecc71'])
plt.ylim(0.55, 1.03)
plt.xticks(rotation=25, ha='right')
plt.title('Train vs. Cross-Validation vs. Test Accuracy (Overfitting & Underfitting Diagnosis)')
plt.tight_layout()
plt.show()

# 2. ROC Curves Comparison
plt.figure(figsize=(8, 6))
for name in ['Support Vector Machine (RBF)', 'Logistic Regression (Tuned L2)', 'Random Forest (n=150)', 'Gradient Boosting', 'Decision Tree (Unconstrained, Overfit)', 'Decision Tree (Depth=1, Underfit)']:
    y_prob = fitted_models[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc_score(y_test, y_prob):.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC-AUC Curve Comparison')
plt.legend(loc='lower right', fontsize=8.5)
plt.tight_layout()
plt.show()

# 3. Confusion Matrix for Top Model (SVM RBF)
best_model = fitted_models['Support Vector Machine (RBF)']
print('Classification Report — Support Vector Machine (RBF):')
print(classification_report(y_test, best_model.predict(X_test), target_names=['Malignant (0)', 'Benign (1)']))